In [2]:
# Cell 1 — Imports
import torch
import torch.nn as nn
from torch.utils.data import (
    Dataset, DataLoader, TensorDataset
)
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizer,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import time
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score
)
from dotenv import load_dotenv
import boto3

load_dotenv()

# Device setup
device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'cpu')

print(f"✅ Device: {device}")
if device.type == 'cuda':
    print(f"   GPU: "
          f"{torch.cuda.get_device_name(0)}")
    print(f"   Memory: "
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print(f"   Running on CPU")
    print(f"   Training will take longer")
    print(f"   Consider using Google Colab")
    print(f"   for GPU acceleration")

# Seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

c:\Users\saich\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Device: cpu
   Running on CPU
   Training will take longer
   Consider using Google Colab
   for GPU acceleration


In [3]:
# Cell 2 — Load from saved tensors
print("Loading preprocessed tensors...")

# Load all splits
train_data = torch.load(
    '../data/processed/train_tensors.pt',
    map_location='cpu')
val_data   = torch.load(
    '../data/processed/val_tensors.pt',
    map_location='cpu')
test_data  = torch.load(
    '../data/processed/test_tensors.pt',
    map_location='cpu')

# Load config
with open(
    '../data/processed/prep_config.json'
) as f:
    config = json.load(f)

BATCH_SIZE  = config['batch_size']
NUM_CLASSES = config['num_classes']
MAX_LENGTH  = config['max_length']
MODEL_NAME  = config['model_name']
LABEL_MAP   = {int(k): v for k, v
               in config['label_map'].items()}

print(f"✅ Tensors loaded")
print(f"   Train: {train_data['input_ids'].shape}")
print(f"   Val:   {val_data['input_ids'].shape}")
print(f"   Test:  {test_data['input_ids'].shape}")
print(f"\n   Model:      {MODEL_NAME}")
print(f"   Classes:    {NUM_CLASSES}")
print(f"   Max length: {MAX_LENGTH}")
print(f"   Batch size: {BATCH_SIZE}")

Loading preprocessed tensors...
✅ Tensors loaded
   Train: torch.Size([108000, 128])
   Val:   torch.Size([12000, 128])
   Test:  torch.Size([7600, 128])

   Model:      distilbert-base-uncased
   Classes:    4
   Max length: 128
   Batch size: 16


In [4]:
# Cell 3 — Create DataLoaders
def make_dataloader(data_dict,
                    batch_size,
                    shuffle=False):
    """Create DataLoader from tensor dict"""
    dataset = TensorDataset(
        data_dict['input_ids'],
        data_dict['attention_mask'],
        data_dict['labels']
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0
    )

train_loader = make_dataloader(
    train_data, BATCH_SIZE, shuffle=True)
val_loader   = make_dataloader(
    val_data,   BATCH_SIZE, shuffle=False)
test_loader  = make_dataloader(
    test_data,  BATCH_SIZE, shuffle=False)

print(f"✅ DataLoaders ready")
print(f"   Training batches:   "
      f"{len(train_loader):,}")
print(f"   Validation batches: "
      f"{len(val_loader):,}")
print(f"   Test batches:       "
      f"{len(test_loader):,}")

✅ DataLoaders ready
   Training batches:   6,750
   Validation batches: 750
   Test batches:       475


In [5]:
# Cell 4 — Load model
print("Loading DistilBERT model...")
print("Downloading pre-trained weights...")

model = DistilBertForSequenceClassification\
    .from_pretrained(
        MODEL_NAME,
        num_labels=NUM_CLASSES,
        output_attentions=False,
        output_hidden_states=False
    )

model = model.to(device)

# Count parameters
total_params    = sum(p.numel()
                     for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad)

print(f"\n✅ Model loaded")
print(f"   Total parameters:     "
      f"{total_params:,}")
print(f"   Trainable parameters: "
      f"{trainable_params:,}")
print(f"\nModel architecture:")
print(model)
# ```

# **What happens when we load the model:**
# ```
# DistilBertForSequenceClassification
# ├── DistilBERT backbone
# │   ├── Embeddings layer
# │   │   (converts token IDs to 768-dim vectors)
# │   ├── 6 Transformer layers
# │   │   (each layer: self-attention + feedforward)
# │   │   Pre-trained weights loaded here ←
# │   └── Output: (batch_size, seq_len, 768)
# │
# └── Classification head (NEW — randomly initialized)
#     ├── Linear(768 → 768)   ← learns task features
#     ├── ReLU activation
#     ├── Dropout(0.2)
#     └── Linear(768 → 4)     ← outputs 4 class scores
#          ↑
#          num_labels=4 — we set this

Loading DistilBERT model...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 938.58it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



✅ Model loaded
   Total parameters:     66,956,548
   Trainable parameters: 66,956,548

Model architecture:
DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), ep

In [6]:
# Cell 5 — Training configuration
LEARNING_RATE = float(os.getenv(
    'LEARNING_RATE', 2e-5))
NUM_EPOCHS    = int(os.getenv(
    'NUM_EPOCHS', 6))
WARMUP_RATIO  = 0.1

total_steps   = len(train_loader) * NUM_EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)

print(f"=== TRAINING CONFIGURATION ===")
print(f"Learning rate:  {LEARNING_RATE}")
print(f"Epochs:         {NUM_EPOCHS}")
print(f"Total steps:    {total_steps:,}")
print(f"Warmup steps:   {warmup_steps:,}")
print(f"Batch size:     {BATCH_SIZE}")

# Optimizer — AdamW
# W = weight decay regularization
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    eps=1e-8,            # numerical stability
    weight_decay=0.01    # L2 regularization
)

# Learning rate scheduler with warmup
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# Loss function
criterion = nn.CrossEntropyLoss()

print(f"\n✅ Optimizer: AdamW "
      f"(lr={LEARNING_RATE})")
print(f"✅ Scheduler: Linear warmup + decay")
print(f"✅ Loss: CrossEntropyLoss")
# ```

# **Why these specific choices:**
# ```
# ADAMW OPTIMIZER:
# Regular Adam adapts learning rate per parameter.
# AdamW adds weight decay (L2 regularization)
# correctly — decoupled from the gradient update.
# This is the standard optimizer for transformer
# fine-tuning. Learning rate 2e-5 is the
# Goldilocks value for BERT — small enough to
# preserve pre-trained knowledge, large enough
# to learn the new task.

# LEARNING RATE WARMUP:
# Start with very small LR → gradually increase
# to target LR → then linearly decay to zero.

# Why warmup?
# Pre-trained weights are carefully calibrated.
# A large LR at the start would destroy them.
# Warmup gives the classification head time to
# initialize before the full LR kicks in.

# Without warmup: catastrophic forgetting —
# model forgets its pre-trained knowledge.

=== TRAINING CONFIGURATION ===
Learning rate:  2e-05
Epochs:         6
Total steps:    40,500
Warmup steps:   4,050
Batch size:     16

✅ Optimizer: AdamW (lr=2e-05)
✅ Scheduler: Linear warmup + decay
✅ Loss: CrossEntropyLoss


In [7]:
# Cell 6 — Training function
def train_epoch(model, loader,
                optimizer, scheduler,
                criterion, device):
    """Train for one epoch"""
    model.train()  # training mode = dropout ON

    total_loss   = 0
    total_correct = 0
    total_samples = 0

    for batch_idx, batch in enumerate(loader):

        input_ids, attention_mask, labels = [
            b.to(device) for b in batch]

        # Zero gradients from previous step
        optimizer.zero_grad()

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        logits = outputs.logits

        # Compute loss
        loss = criterion(logits, labels)

        # Backward pass — compute gradients
        loss.backward()

        # Clip gradients to prevent explosion
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        # Update weights
        optimizer.step()

        # Update learning rate
        scheduler.step()

        # Track metrics
        total_loss    += loss.item()
        preds          = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

        # Log every 100 batches
        if (batch_idx + 1) % 100 == 0:
            avg_loss = total_loss / (batch_idx + 1)
            acc      = total_correct / total_samples
            print(f"  Batch {batch_idx+1:>5,}/"
                  f"{len(loader):,} | "
                  f"Loss: {avg_loss:.4f} | "
                  f"Acc: {acc:.4f}")

    return (total_loss / len(loader),
            total_correct / total_samples)

In [8]:
# Cell 7 — Evaluation function
def evaluate(model, loader,
             criterion, device):
    """Evaluate model on a data loader"""
    model.eval()  # eval mode = dropout OFF

    total_loss    = 0
    all_preds     = []
    all_labels    = []

    with torch.no_grad():  # no gradient tracking
        for batch in loader:

            input_ids, attention_mask, labels = [
                b.to(device) for b in batch]

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            logits = outputs.logits
            loss   = criterion(logits, labels)

            total_loss += loss.item()
            preds       = logits.argmax(dim=1)

            all_preds.extend(
                preds.cpu().numpy())
            all_labels.extend(
                labels.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(
        all_labels, all_preds)
    f1       = f1_score(
        all_labels, all_preds,
        average='macro')

    return avg_loss, accuracy, f1, \
           all_preds, all_labels

In [9]:
# Cell 8 — Main training loop
print("="*55)
print("  STARTING DISTILBERT FINE-TUNING")
print("="*55)
print(f"  Epochs:    {NUM_EPOCHS}")
print(f"  Device:    {device}")
print(f"  Est. time: ~30-60 min on CPU")
print(f"             ~5-10 min on GPU")
print("="*55)

os.makedirs('../models', exist_ok=True)
os.makedirs('../logs', exist_ok=True)

# Track history
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss':   [], 'val_acc':   [],
    'val_f1':     [], 'lr':        []
}

best_val_f1   = 0
best_epoch    = 0
training_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()

    print(f"\n{'─'*55}")
    print(f"  EPOCH {epoch}/{NUM_EPOCHS}")
    print(f"{'─'*55}")

    # ── Training ──────────────────────────────
    print("  Training...")
    train_loss, train_acc = train_epoch(
        model, train_loader,
        optimizer, scheduler,
        criterion, device)

    # ── Validation ────────────────────────────
    print("  Evaluating on validation set...")
    val_loss, val_acc, val_f1, _, _ = evaluate(
        model, val_loader, criterion, device)

    epoch_time = time.time() - epoch_start
    current_lr = scheduler.get_last_lr()[0]

    # Store history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    history['lr'].append(current_lr)

    print(f"\n  Results Epoch {epoch}:")
    print(f"    Train Loss: {train_loss:.4f} | "
          f"Train Acc: {train_acc:.4f}")
    print(f"    Val Loss:   {val_loss:.4f} | "
          f"Val Acc:   {val_acc:.4f} | "
          f"Val F1: {val_f1:.4f}")
    print(f"    LR: {current_lr:.2e} | "
          f"Time: {epoch_time:.1f}s")

    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch  = epoch
        model.save_pretrained(
            '../models/best_distilbert')
        print(f"    ✅ New best model saved! "
              f"F1={val_f1:.4f}")

total_time = time.time() - training_start
print(f"\n{'='*55}")
print(f"  TRAINING COMPLETE")
print(f"  Total time: {total_time/60:.1f} min")
print(f"  Best epoch: {best_epoch}")
print(f"  Best val F1: {best_val_f1:.4f}")
print(f"{'='*55}")

# Save training history
with open('../logs/training_history.json',
          'w') as f:
    json.dump(history, f, indent=4)

  STARTING DISTILBERT FINE-TUNING
  Epochs:    6
  Device:    cpu
  Est. time: ~30-60 min on CPU
             ~5-10 min on GPU

───────────────────────────────────────────────────────
  EPOCH 1/6
───────────────────────────────────────────────────────
  Training...
  Batch   100/6,750 | Loss: 1.3841 | Acc: 0.2556
  Batch   200/6,750 | Loss: 1.3743 | Acc: 0.3053
  Batch   300/6,750 | Loss: 1.3434 | Acc: 0.4235
  Batch   400/6,750 | Loss: 1.2655 | Acc: 0.5291
  Batch   500/6,750 | Loss: 1.1580 | Acc: 0.5975
  Batch   600/6,750 | Loss: 1.0514 | Acc: 0.6455
  Batch   700/6,750 | Loss: 0.9656 | Acc: 0.6778
  Batch   800/6,750 | Loss: 0.8892 | Acc: 0.7059
  Batch   900/6,750 | Loss: 0.8265 | Acc: 0.7282
  Batch 1,000/6,750 | Loss: 0.7775 | Acc: 0.7451
  Batch 1,100/6,750 | Loss: 0.7357 | Acc: 0.7595
  Batch 1,200/6,750 | Loss: 0.7024 | Acc: 0.7708
  Batch 1,300/6,750 | Loss: 0.6720 | Acc: 0.7810
  Batch 1,400/6,750 | Loss: 0.6440 | Acc: 0.7904
  Batch 1,500/6,750 | Loss: 0.6227 | Acc: 0.7977

KeyboardInterrupt: 

In [ ]:
# Cell 9 — Training visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_range = range(1, NUM_EPOCHS + 1)

# Loss curves
axes[0].plot(epochs_range, history['train_loss'],
             'o-', color='#2196F3',
             label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'],
             'o--', color='#F44336',
             label='Val Loss', linewidth=2)
axes[0].set_title('Loss Curves',
                   fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('CrossEntropy Loss')
axes[0].legend()
axes[0].set_xticks(epochs_range)

# Accuracy curves
axes[1].plot(epochs_range, history['train_acc'],
             'o-', color='#2196F3',
             label='Train Acc', linewidth=2)
axes[1].plot(epochs_range, history['val_acc'],
             'o--', color='#F44336',
             label='Val Acc', linewidth=2)
axes[1].set_title('Accuracy Curves',
                   fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].set_xticks(epochs_range)

# F1 + Learning Rate
ax_f1 = axes[2]
ax_lr = ax_f1.twinx()

ax_f1.plot(epochs_range, history['val_f1'],
           'o-', color='#4CAF50',
           label='Val F1', linewidth=2)
ax_lr.plot(epochs_range, history['lr'],
           's--', color='#FF9800',
           label='Learning Rate',
           linewidth=1.5, alpha=0.7)

ax_f1.set_title('Validation F1 + Learning Rate',
                fontweight='bold')
ax_f1.set_xlabel('Epoch')
ax_f1.set_ylabel('F1 Score', color='#4CAF50')
ax_lr.set_ylabel('Learning Rate',
                  color='#FF9800')
ax_f1.set_xticks(epochs_range)

lines1, labels1 = ax_f1.get_legend_handles_labels()
lines2, labels2 = ax_lr.get_legend_handles_labels()
ax_f1.legend(lines1+lines2,
             labels1+labels2, loc='lower right')

plt.suptitle('DistilBERT Fine-Tuning History',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../logs/training_curves.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 10 — Load best model and evaluate on test
print("Loading best model for final evaluation...")

best_model = DistilBertForSequenceClassification\
    .from_pretrained('../models/best_distilbert')
best_model = best_model.to(device)

print("Evaluating on test set...")
test_loss, test_acc, test_f1, \
    test_preds, test_labels = evaluate(
        best_model, test_loader,
        criterion, device)

print(f"\n{'='*55}")
print(f"  FINAL TEST RESULTS")
print(f"{'='*55}")
print(f"  Test Loss:     {test_loss:.4f}")
print(f"  Test Accuracy: {test_acc:.4f} "
      f"({test_acc*100:.2f}%)")
print(f"  Test F1:       {test_f1:.4f}")
print(f"{'='*55}")

# Detailed classification report
label_names = ['World', 'Sports',
               'Business', 'Sci/Tech']
print(f"\n{classification_report(test_labels, test_preds, target_names=label_names)}")

In [ ]:
# Cell 11 — Confusion matrix
plt.figure(figsize=(8, 6))

cm = confusion_matrix(test_labels, test_preds)
cm_pct = cm.astype('float') / \
         cm.sum(axis=1)[:, np.newaxis] * 100

sns.heatmap(
    cm_pct,
    annot=True,
    fmt='.1f',
    cmap='Blues',
    xticklabels=label_names,
    yticklabels=label_names,
    cbar_kws={'label': 'Percentage (%)'}
)
plt.title(f'DistilBERT Confusion Matrix\n'
          f'Test Accuracy: {test_acc*100:.2f}% | '
          f'F1: {test_f1:.4f}',
          fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('../logs/confusion_matrix.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Print confusion insights
print("\n=== CONFUSION INSIGHTS ===")
for i, true_label in enumerate(label_names):
    misclassified = [(label_names[j], cm[i][j])
                     for j in range(len(label_names))
                     if i != j and cm[i][j] > 0]
    if misclassified:
        worst = max(misclassified,
                    key=lambda x: x[1])
        print(f"  {true_label:<12} → "
              f"most confused with "
              f"{worst[0]} ({worst[1]} cases)")

In [ ]:
# Cell 12 — Per-category deep dive
print("=== PER-CATEGORY ANALYSIS ===\n")

for i, category in enumerate(label_names):
    mask     = test_labels == i
    cat_acc  = (test_preds[mask] == i).mean()
    cat_count = mask.sum()

    print(f"{category}:")
    print(f"  Test samples: {cat_count:,}")
    print(f"  Accuracy:     {cat_acc:.4f} "
          f"({cat_acc*100:.1f}%)")

    # What does it get confused with?
    wrong_mask  = (test_labels == i) & \
                  (test_preds != i)
    wrong_preds = test_preds[wrong_mask]
    if len(wrong_preds) > 0:
        from collections import Counter
        confused_with = Counter(wrong_preds)
        top_confusion = confused_with.most_common(1)[0]
        confused_label = label_names[top_confusion[0]]
        print(f"  Confused with: {confused_label} "
              f"({top_confusion[1]} times)")
    print()

In [ ]:
# Cell 13 — Save all results
final_results = {
    'model_name':         MODEL_NAME,
    'num_epochs':         NUM_EPOCHS,
    'best_epoch':         best_epoch,
    'best_val_f1':        round(best_val_f1, 4),
    'test_accuracy':      round(test_acc, 4),
    'test_f1_macro':      round(test_f1, 4),
    'test_loss':          round(test_loss, 4),
    'training_config': {
        'learning_rate':  LEARNING_RATE,
        'batch_size':     BATCH_SIZE,
        'max_length':     MAX_LENGTH,
        'optimizer':      'AdamW',
        'scheduler':      'linear_warmup',
        'warmup_ratio':   WARMUP_RATIO
    },
    'per_class_accuracy': {
        label_names[i]: round(
            float((test_preds[test_labels==i]
                   == i).mean()), 4)
        for i in range(NUM_CLASSES)
    }
}

with open('../logs/bert_results.json',
          'w') as f:
    json.dump(final_results, f, indent=4)

print("✅ Results saved to logs/bert_results.json")
print(json.dumps(final_results, indent=4))